# Polymarket réagit-il aux qualifications de F1 ?

*Projet Python pour la data science - ENSAE 2A, 2026-2027 - Auteurs : Orane Bertout, Stanislas Hanriot, Colombe De Gélis*

Polymarket est un marché de prédiction : pour chaque Grand Prix, il existe un contrat par pilote qui rapporte 1 $ si ce pilote gagne. Le prix du contrat peut donc se lire comme la probabilité de victoire estimée par le marché.

Les qualifications, la veille de la course, fixent la grille de départ. Sur un circuit où il est difficile de dépasser (Monaco par exemple), partir devant décide presque de la course. 

**Question : le marché réagit-il plus fortement aux qualifications sur les circuits où il est difficile de dépasser ?**

Données utilisées :
- Polymarket : prix minute par minute des 24 Grands Prix de 2025 ;
- OpenF1 : horaires des séances 2025 et dépassements enregistrés pendant les courses 2023-2024;
- Jolpica : calendrier et coordonnées des circuits ;

In [10]:
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf

from src.config import ANNEE, ANNEE_COEFF

from src.collect import charger, collecter_polymarket, collecter_openf1, collecter_jolpica
from src.depassement import coef_depassement
from src.traitement import nettoyer_marches, nettoyer_prix, table_grands_prix


## 1. Récupération des données

La première étape consiste à charger les données qui seront utilisés plus tard.

In [11]:
events = charger("polymarket_f1_events.parquet", collecter_polymarket, ANNEE)
marches = charger("polymarket_f1_markets.parquet", collecter_polymarket, ANNEE)
prix = charger("polymarket_f1_prices.parquet", collecter_polymarket, ANNEE)
seances = charger("openf1_sessions.parquet", collecter_openf1, ANNEE)
calendrier = charger("jolpica_calendrier.parquet", collecter_jolpica, ANNEE)
coef = charger("coefficients_depassement.parquet", coef_depassement, ANNEE_COEFF)
courses_dep = charger("depassements_par_course.parquet", coef_depassement, ANNEE_COEFF)

print(len(events), "GP,", len(marches), "marchés,", len(prix), "prix")

24 Grands Prix trouvés
  ok : F1 Singapore Grand Prix Winner
  ok : F1 Azerbaijan Grand Prix Winner
  ok : Mexican Grand Prix Winner
  ok : United States Grand Prix Winner
  ok : F1 Hungarian Grand Prix Winner
  ok : Japan Grand Prix Winner
  ok : Australia Grand Prix Winner
  ok : Monaco Grand Prix Winner
  ok : F1 Belgian Grand Prix Winner
  ok : Italy Grand Prix Winner
  ok : F1 British Grand Prix Winner
  ok : F1 Austrian Grand Prix Winner
  ok : Spanish Grand Prix Winner
  ok : F1 Dutch Grand Prix Winner
  ok : Miami Grand Prix Winner
  ok : F1 Italian Grand Prix Winner
  ok : F1 Canadian Grand Prix Winner
  ok : China Grand Prix Winner
  ok : Bahrain Grand Prix Winner
  ok : Saudi Arabian Grand Prix Winner
  ok : Qatar Grand Prix: Driver Winner
  ok : Brazilian Grand Prix: Driver Winner
  ok : Las Vegas Grand Prix: Driver Winner
  ok : Abu Dhabi Grand Prix: Driver Winner
24 GP, 622 marchés, 6315717 prix


## 2. Nettoyage des données

Les noms de pilotes ne sont pas toujours écrits pareil d'un GP à l'autre (« Hadjar », « Isack Hadjar », « George Russel »…) et Polymarket ajoute des marchés de réserve (« Other », « Driver A »…). On ramène chaque nom au nom de famille du pilote et on supprime les marchés de réserve.

In [12]:
marches = nettoyer_marches(marches)
prix = nettoyer_prix(prix, marches)
marches.groupby("event_id")["pilote"].count().value_counts()

138 marchés de réserve supprimés


pilote
20    20
21     4
Name: count, dtype: int64

Selon les résultats renvoyés, 138 marchés de réserve ont été supprimés. \n
20 GPs avaient 20 pilotes participants et 4 en avaient 21. (Il n'y avait pas réellement 21 pilotes sur la grille de départ mais un changement a eu lieu).

Contrôle de la qualité des prix : valeurs manquantes, doublons, valeurs hors de [0, 1] et nombre de points de prix par GP.

In [13]:
print("Valeurs manquantes :", prix.isna().sum().sum())
print("Doublons :", prix.duplicated(["event_id", "pilote", "timestamp_utc"]).sum())
print("Prix hors de [0, 1] :", (~prix["price"].between(0, 1)).sum())
prix.groupby("event_id").size().describe()

Valeurs manquantes : 0
Doublons : 0
Prix hors de [0, 1] : 0


count        24.000000
mean     259845.666667
std      112060.428355
min      110561.000000
25%      178926.500000
50%      192602.500000
75%      352369.250000
max      519125.000000
dtype: float64

Selon les résultats renvoyés, il n'y avait ni valeurs manquantes, ni doublons et ni prix hors de [0, 1].\n
Le tableau donne des informations sur les points de prix. Ainsi, sur les 24 GPs, la moyenne est d'environ 260 000 point de prix avec un minimum autour de 110 560, cela reste donc largement exploitable.

On rassemble ensuite les sources dans une table avec une ligne par Grand Prix : horaires des qualifications et de la course (OpenF1), circuit (Jolpica) et coefficient de dépassement. Toutes les heures sont en UTC dans les trois sources.

In [14]:
gp = table_grands_prix(events, marches, seances, calendrier)
assert gp["circuit_id"].notna().all(), "GP sans circuit : vérifier la jointure sur la date"

gp = gp.merge(coef, on="circuit_id", how="left")
print("GP sans coefficient de dépassement :", gp["coef_depassement"].isna().sum())
gp[["title", "circuit_id", "quali_debut", "course_debut", "sprint", "coef_depassement"]]

GP sans coefficient de dépassement : 0


,title,circuit_id,quali_debut,course_debut,sprint,coef_depassement
0,Australia Grand Prix Winner,albert_park,2025-03-15 05:00:00+00:00,2025-03-16 04:00:00+00:00,False,211.0
1,China Grand Prix Winner,shanghai,2025-03-22 07:00:00+00:00,2025-03-23 07:00:00+00:00,True,310.0
2,Japan Grand Prix Winner,suzuka,2025-04-05 06:00:00+00:00,2025-04-06 05:00:00+00:00,False,236.0
3,Bahrain Grand Prix Winner,bahrain,2025-04-12 16:00:00+00:00,2025-04-13 15:00:00+00:00,False,414.0
4,Saudi Arabian Grand Prix Winner,jeddah,2025-04-19 17:00:00+00:00,2025-04-20 17:00:00+00:00,False,108.0
5,Miami Grand Prix Winner,miami,2025-05-03 20:00:00+00:00,2025-05-04 20:00:00+00:00,True,201.0
6,Italy Grand Prix Winner,imola,2025-05-17 14:00:00+00:00,2025-05-18 13:00:00+00:00,False,133.0
7,Monaco Grand Prix Winner,monaco,2025-05-24 14:00:00+00:00,2025-05-25 13:00:00+00:00,False,47.0
8,Spanish Grand Prix Winner,catalunya,2025-05-31 14:00:00+00:00,2025-06-01 13:00:00+00:00,False,334.0
9,F1 Canadian Grand Prix Winner,villeneuve,2025-06-14 20:00:00+00:00,2025-06-15 18:00:00+00:00,False,206.0


## 3. Coefficient de dépassement

Pour mesurer la difficulté de dépassement, on compte le nombre de dépassement effectuer durant la course. 
Ces données sont récupérées sur les éditions des années 2023 et 2024 et non de 2025 afin de ne pas mesurer le coefficient sur les mêmes courses que la réaction du marché.

In [15]:
print("Courses 2023-2024 sans données de dépassements :", courses_dep["depassements"].isna().sum())
print("Circuits par nombre d'éditions utilisées :")
print(coef["nb_courses"].value_counts().sort_index())
coef.sort_values("coef_depassement")

Courses 2023-2024 sans données de dépassements : 0
Circuits par nombre d'éditions utilisées :
nb_courses
1    24
Name: count, dtype: int64


,circuit_id,coef_depassement,depassement_par_tour,nb_courses
12,monaco,47.0,0.602564,1
8,jeddah,108.0,2.160000,1
17,silverstone,120.0,2.307692,1
6,imola,133.0,2.111111,1
3,baku,178.0,3.490196,1
7,interlagos,183.0,2.652174,1
1,americas,201.0,3.589286,1
11,miami,201.0,3.526316,1
21,villeneuve,206.0,2.942857,1
15,rodriguez,206.0,2.901408,1


Ainsi, on peut voir que Monaco est le circuit le moins propice au dépassement de loin ce qui est bien cohérent avec ce qui est observé en F1 alors que de l'autre coté les circuits de Las Vegas, Catalunya et Bahrain sont les plus propices. Cela reste toujours cohérent.

### Tests

Tests réalisés pour vérifier que les fonctions du fichiers dépassements fonctionnent correctement. Pour les lancer, il suffit de les retirer des commentaires.

In [16]:
"""
from tests.test_depassement import test_depassement, test_courses, test_coef_depassement

test_depassement()
test_courses()
test_coef_depassement()
"""

'\nfrom tests.test_depassement import test_depassement, test_courses, test_coef_depassement\n\ntest_depassement()\ntest_courses()\ntest_coef_depassement()\n'